# Notebook 1 — Data Loading

**Assam Government Procurement Analytics**

This notebook is the first step in the pipeline. Its only job is to take the raw CSV exports and get them into a proper relational database so every later notebook can query them with SQL instead of re-parsing CSVs.

**What this notebook does:**
1. Reads the two useful raw files — `main.csv` (core tender records) and `tender_milestones.csv` (milestone dates).
2. Creates a SQLite database at `db/assam_procurement.db`.
3. Writes the two files into tables `tenders` and `milestones`.
4. Runs verification queries (row counts + sample rows) to confirm the load was correct.

No cleaning happens here — this is a raw, faithful load. Cleaning is handled in Notebook 2.

`tender_documents.csv` and `tender_participationFee.csv` are excluded — they aren't useful for this analysis.

## Step 1 — Imports and file paths

We use `pandas` to read the CSVs and `sqlite3` (Python's built-in library) to create and write to the database. Paths are defined relative to this notebook's location (`notebooks/`), pointing up one level to the project root.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

# Project paths (this notebook lives in notebooks/, so we go up one level)
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_DIR = PROJECT_ROOT / "db"
DB_PATH = DB_DIR / "assam_procurement.db"

MAIN_CSV = RAW_DIR / "main.csv"
MILESTONES_CSV = RAW_DIR / "tender_milestones.csv"

# Make sure the db/ folder exists before we try to write a database into it
DB_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:      ", PROJECT_ROOT)
print("Main CSV:           ", MAIN_CSV, "| exists:", MAIN_CSV.exists())
print("Milestones CSV:     ", MILESTONES_CSV, "| exists:", MILESTONES_CSV.exists())
print("Database will be at:", DB_PATH)

Project root:       C:\Users\Avni\assam-procurement-analytics
Main CSV:            C:\Users\Avni\assam-procurement-analytics\data\raw\main.csv | exists: True
Milestones CSV:      C:\Users\Avni\assam-procurement-analytics\data\raw\tender_milestones.csv | exists: True
Database will be at: C:\Users\Avni\assam-procurement-analytics\db\assam_procurement.db


## Step 2 — Load the raw CSVs into pandas

Before writing anything to SQLite, we load both files into DataFrames and take a quick look at their shape. This is a raw read — no `dtype` coercion or cleaning yet, so that the database faithfully mirrors the source files. `low_memory=False` avoids pandas' mixed-dtype warning on the wider `main.csv` file.

In [2]:
df_main = pd.read_csv(MAIN_CSV, low_memory=False)
df_milestones = pd.read_csv(MILESTONES_CSV, low_memory=False)

print("main.csv               ->", df_main.shape[0], "rows,", df_main.shape[1], "columns")
print("tender_milestones.csv   ->", df_milestones.shape[0], "rows,", df_milestones.shape[1], "columns")

main.csv               -> 34232 rows, 26 columns
tender_milestones.csv   -> 34232 rows, 9 columns


## Step 3 — Write to SQLite

We open a connection to `db/assam_procurement.db` (SQLite creates the file automatically if it doesn't exist yet) and write each DataFrame to its own table using `DataFrame.to_sql`:

- `main.csv` → table **`tenders`**
- `tender_milestones.csv` → table **`milestones`**

`if_exists=\"replace\"` makes this notebook safely re-runnable — running it again rebuilds the tables from scratch instead of appending duplicate rows.

In [3]:
conn = sqlite3.connect(DB_PATH)

df_main.to_sql("tenders", conn, if_exists="replace", index=False)
df_milestones.to_sql("milestones", conn, if_exists="replace", index=False)

conn.commit()
print("Tables written to", DB_PATH)

Tables written to C:\Users\Avni\assam-procurement-analytics\db\assam_procurement.db


## Step 4 — Verification

Now we check the load actually worked, using real SQL queries executed through `sqlite3` (not pandas methods). For each table we confirm:

1. The row count matches the source CSV.
2. A sample of rows looks sane (correct columns, no obvious corruption).

Both tables should show **34,232 rows**, matching the row counts of `main.csv` and `tender_milestones.csv`.

In [4]:
cursor = conn.cursor()

# --- Row counts, via real SQL ---
cursor.execute("SELECT COUNT(*) FROM tenders;")
tenders_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM milestones;")
milestones_count = cursor.fetchone()[0]

print("tenders table row count:   ", tenders_count)
print("milestones table row count:", milestones_count)

tenders table row count:    34232
milestones table row count: 34232


Row counts match, but that alone doesn't rule out corrupted or misaligned values. Next, we pull a few sample rows from each table to eyeball the columns and confirm the data looks correct.

In [5]:
# --- Sample rows, via real SQL ---
# pd.read_sql_query just hands the SQL string to sqlite3 and wraps the
# result as a DataFrame for nicer display — the query itself is plain SQL.
sample_tenders_sql = "SELECT * FROM tenders LIMIT 5;"
sample_milestones_sql = "SELECT * FROM milestones LIMIT 5;"

sample_tenders = pd.read_sql_query(sample_tenders_sql, conn)
sample_milestones = pd.read_sql_query(sample_milestones_sql, conn)

print("Sample rows from 'tenders':")
display(sample_tenders)

print("\nSample rows from 'milestones':")
display(sample_milestones)

Sample rows from 'tenders':


,_link,id,tag,date,ocid,Payment Mode,initiationType,fiscal_year,buyer_name,tender_id,...,tender_mainProcurementCategory,tender_contractType,tender_numberOfTenderers,tender_datePublished,tender_allowPreferentialBidder,tender_externalReference,tender_value_amount,tender_bidOpening_date,tender_tenderPeriod_durationInDays,tenderclassification_description
0,id-0.0,ocds-kjhdrl-2016_DOT_946_1-2022-09-29,compiled,2022-09-29,ocds-kjhdrl-2016_DOT_946_1,Offline,tender,2016-2017,Department of Tourism,2016_DOT_946_1,...,Works,Item Rate,5.0,2016-06-16 12:00:00,No,ATDC/CS/558/2016/1036_E002,25132914,12-07-2016 12:30,180,Composite Works
1,id-0.1,ocds-kjhdrl-2016_DoWR_1302_1-2022-09-29,compiled,2022-09-29,ocds-kjhdrl-2016_DoWR_1302_1,Offline,tender,2016-2017,Department of Water Resources,2016_DoWR_1302_1,...,Works,Works,7.0,2016-12-12 18:00:00,No,Dib/SDRF/2016-17/6,25349923,02-01-2017 14:00,180,Civil Works
2,id-0.2,ocds-kjhdrl-2016_DoWR_1318_1-2022-09-29,compiled,2022-09-29,ocds-kjhdrl-2016_DoWR_1318_1,Offline,tender,2016-2017,Department of Water Resources,2016_DoWR_1318_1,...,Works,Works,5.0,2016-12-15 9:00:00,No,Bak/SDRF/2016-17/3,7967277,04-01-2017 14:05,180,Civil Works
3,id-0.3,ocds-kjhdrl-2016_DoWR_1334_1-2022-09-29,compiled,2022-09-29,ocds-kjhdrl-2016_DoWR_1334_1,Offline,tender,2016-2017,Department of Water Resources,2016_DoWR_1334_1,...,Works,Works,10.0,2016-12-28 18:00:00,No,Maj/SDRF/2016-17/3,9803783,11-01-2017 14:05,180,Civil Works
4,id-0.4,ocds-kjhdrl-2016_DoWR_1366_1-2022-09-29,compiled,2022-09-29,ocds-kjhdrl-2016_DoWR_1366_1,Offline,tender,2016-2017,Department of Water Resources,2016_DoWR_1366_1,...,Works,Works,10.0,2016-12-22 18:00:00,No,Siv/SDRF/2016-17/3,12441168,11-01-2017 14:00,180,Civil Works



Sample rows from 'milestones':


,_link,_link_main,code,type,title,type.1,dueDate,title.1,dueDate.1
0,id-0.0.tender.milestones.0,id-0.0,PreBid Meeting Date,assessment,Price Bid Opening Date,assessment,None,Price Bid Opening Date,None
1,id-0.1.tender.milestones.0,id-0.1,PreBid Meeting Date,assessment,Price Bid Opening Date,assessment,None,Price Bid Opening Date,None
2,id-0.2.tender.milestones.0,id-0.2,PreBid Meeting Date,assessment,Price Bid Opening Date,assessment,None,Price Bid Opening Date,None
3,id-0.3.tender.milestones.0,id-0.3,PreBid Meeting Date,assessment,Price Bid Opening Date,assessment,None,Price Bid Opening Date,None
4,id-0.4.tender.milestones.0,id-0.4,PreBid Meeting Date,assessment,Price Bid Opening Date,assessment,None,Price Bid Opening Date,None


We also check the table schemas with `PRAGMA table_info(...)`, a SQLite-specific SQL statement that lists every column and its inferred type — a useful sanity check that all 26 columns of `main.csv` and all 9 columns of `tender_milestones.csv` made it into the database.

In [6]:
tenders_schema = pd.read_sql_query("PRAGMA table_info(tenders);", conn)
milestones_schema = pd.read_sql_query("PRAGMA table_info(milestones);", conn)

print(f"'tenders' table:    {len(tenders_schema)} columns (expected 26)")
print(f"'milestones' table: {len(milestones_schema)} columns (expected 9)")

assert len(tenders_schema) == 26, "Unexpected column count in 'tenders'"
assert len(milestones_schema) == 9, "Unexpected column count in 'milestones'"
assert tenders_count == 34232, "Unexpected row count in 'tenders'"
assert milestones_count == 34232, "Unexpected row count in 'milestones'"

print("\nAll checks passed — load verified.")

'tenders' table:    26 columns (expected 26)
'milestones' table: 9 columns (expected 9)

All checks passed — load verified.


## Step 5 — Close the connection

We close the SQLite connection now that all writes are committed. Every later notebook opens its own fresh connection to `db/assam_procurement.db`.

In [7]:
conn.close()
print("Connection closed.")

Connection closed.


## Summary

- Loaded `data/raw/main.csv` (34,232 rows × 26 columns) into SQLite table **`tenders`**.
- Loaded `data/raw/tender_milestones.csv` (34,232 rows × 9 columns) into SQLite table **`milestones`**.
- Database created at `db/assam_procurement.db`.
- Row counts and schemas verified against the source CSVs — no data lost in the load.
- `tender_documents.csv` and `tender_participationFee.csv` were intentionally skipped (not useful for this analysis).

This was a **raw load only** — known data quality issues (fully-null columns, mixed date formats, comma-formatted numbers in `tender_value_amount`, etc.) are untouched and will be addressed in **Notebook 2 — Data Cleaning**.